# EDA — CineMatch | MovieLens 32M
Análisis exploratorio del dataset público MovieLens 32M.
El objetivo es verificar la calidad de los datos, detectar anomalías 
y preparar el terreno para la fase de limpieza.

### 1 - Carga de datos 

Cargamos los 4 ficheros CSV del dataset MoviLens 32M desde "data/raw".
Los resultados esperados segun la documentacion oficial son : 
- ratings: 32.000.204 filas × 4 columnas
- movies: 87.585 filas × 3 columnas
- links: 87.585 filas × 3 columnas
- tags: 2.000.072 filas × 4 columnas

In [13]:
import pandas as pd
import os

# Ruta relativa desde la carpeta del notebook hacia data/raw/
BASE_PATH = os.path.join(os.path.dirname(os.getcwd()), "..", "data", "raw")

ratings = pd.read_csv(os.path.join(BASE_PATH, "ratings.csv"))
movies  = pd.read_csv(os.path.join(BASE_PATH, "movies.csv"))
links   = pd.read_csv(os.path.join(BASE_PATH, "links.csv"))
tags    = pd.read_csv(os.path.join(BASE_PATH, "tags.csv"))

print("ratings:", ratings.shape)
print("movies: ", movies.shape)
print("links:  ", links.shape)
print("tags:   ", tags.shape)

ratings: (32000204, 4)
movies:  (87585, 3)
links:   (87585, 3)
tags:    (2000072, 4)


### 2. Calidad de los datos - nulos y tipos 

Verificamos que no hay valores nulos en los campos clave y que los tipos de datos son correctos 

In [6]:
for nombre, df in [("ratings", ratings),("movies", movies),("links", links), ("tags", tags)]:
    print(f"=== {nombre} ===")
    print(df.dtypes)
    print("Nulos:\n", df.isnull().sum())
    print()

=== ratings ===
userId         int64
movieId        int64
rating       float64
timestamp      int64
dtype: object
Nulos:
 userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

=== movies ===
movieId     int64
title      object
genres     object
dtype: object
Nulos:
 movieId    0
title      0
genres     0
dtype: int64

=== links ===
movieId      int64
imdbId       int64
tmdbId     float64
dtype: object
Nulos:
 movieId      0
imdbId       0
tmdbId     124
dtype: int64

=== tags ===
userId        int64
movieId       int64
tag          object
timestamp     int64
dtype: object
Nulos:
 userId        0
movieId       0
tag          17
timestamp     0
dtype: int64



### 3. Primera vista de cada fichero

In [10]:
print("=== ratings ===")
display(ratings.head(3))

print("=== movies ===")
display(movies.head(3))

print("=== links ===")
display(links.head(3))

print("=== tags ===")
display(tags.head(3))

=== ratings ===


,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976


=== movies ===


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance


=== links ===


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0


=== tags ===


,userId,movieId,tag,timestamp
0,22,26479,Kevin Kline,1583038886
1,22,79592,misogyny,1581476297
2,22,247150,acrophobia,1622483469


In [11]:
# Inspeccionamos todos los casos anómalos detectados

# 1. Películas sin tmdbId — no podrán enriquecerse con TMDB API
peliculas_sin_tmdb = links[links["tmdbId"].isna()]
print(f"Películas sin tmdbId: {len(peliculas_sin_tmdb)}")
sin_tmdb_info = peliculas_sin_tmdb.merge(movies, on="movieId")
display(sin_tmdb_info.head(10))

# 2. Tags con etiqueta vacía — se eliminarán en la limpieza
tags_nulos = tags[tags["tag"].isna()]
print(f"\nTags con etiqueta vacía: {len(tags_nulos)}")
display(tags_nulos)

# 3. tmdbId viene como float64 por los nulos — lo convertimos a Int64 (nullable integer)
links["tmdbId"] = links["tmdbId"].astype("Int64")
print(f"\ntmdbId convertido a: {links['tmdbId'].dtype}")

Películas sin tmdbId: 124


,movieId,imdbId,tmdbId,title,genres
0,721,114103,NaN,Halfmoon (Paul Bowles - Halbmond) (1995),Drama
1,730,125877,NaN,Low Life (1994),Drama
2,770,38426,NaN,Costa Brava (1946),Drama
3,791,113610,NaN,"Last Klezmer: Leopold Kozlowski, His Life and ...",Documentary
4,1107,102336,NaN,Loser (1991),Comedy
5,1142,116403,NaN,Get Over It (1996),Drama
6,1316,115548,NaN,Anna (1996),Drama
7,1421,113212,NaN,Grateful Dead (1995),Documentary
8,1434,123281,NaN,"Stranger, The (1994)",Action
9,1630,123953,NaN,"Lay of the Land, The (1997)",Comedy|Drama



Tags con etiqueta vacía: 17


,userId,movieId,tag,timestamp
185377,27046,33826,NaN,1221450908
1394089,89369,281500,NaN,1670942104
1914668,153443,123,NaN,1199450867
1914669,153443,346,NaN,1199451946
1914673,153443,1184,NaN,1199452261
1914680,153443,1785,NaN,1199452006
1914681,153443,2194,NaN,1199450677
1914683,153443,2691,NaN,1199451002
1914691,153443,4103,NaN,1199451920
1914693,153443,4473,NaN,1199451040



tmdbId convertido a: Int64


### 4. Observaciones de calidad 

- `ratings`: sin nulos, tipos correctos.
- `movies`: sin nulos. Los géneros vienen separados por `|` → se normalizarán en la fase de limpieza.
- `links`: 124 nulos en `tmdbId`. Estas películas no tendrán poster ni sinopsis via TMDB API. Se gestionarán mostrando solo datos locales como plan B.
- `tags`: 17 nulos en la columna `tag`. Son registros donde el usuario dejó la etiqueta vacía → se eliminarán en la fase de limpieza.
- `tmdbId` aparece como `float64` en vez de `int` porque pandas convierte automáticamente columnas con nulos.